# 2リンクアームの運動学(演習)

解説資料は `research-handbook/robotics/kinematics.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/arm_kinematics_solution.ipynb` にあるが、まず自力で取り組むこと。

**このnotebookで学ぶこと**

- 2リンク平面アームの順運動学(FK)と到達可能領域
- 解析的逆運動学(IK):余弦定理と atan2、elbow-up/down の2解
- ヤコビアンの導出・数値微分による検算・特異点
- 擬似逆行列による数値IKで手先軌道を追従させる

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

L1, L2 = 1.0, 0.8   # リンク長 [m]

## 課題1: 順運動学

$$x = l_1 \cos q_1 + l_2 \cos(q_1 + q_2), \qquad y = l_1 \sin q_1 + l_2 \sin(q_1 + q_2)$$

肘の位置も返すようにしておくと、後でアームの描画に使えます。

In [ ]:
def forward_kinematics(q, l1=L1, l2=L2):
    """課題1: 順運動学。
    q = [q1, q2] [rad] から (肘位置 p1, 手先位置 p2) を返す"""
    q1, q2 = q
    # ---- ここから課題1 ----
    # kinematics.md の式の通り。肘は l1 だけ、手先はさらに l2 だけ先
    p1 = None
    p2 = None
    # ---- ここまで ----
    return p1, p2

# 動作確認: q1=0, q2=0 なら腕は真横に伸び、手先は (l1+l2, 0)
p1, p2 = forward_kinematics([0.0, 0.0])
print("肘:", p1, " 手先:", p2)
p1, p2 = forward_kinematics([np.pi / 2, -np.pi / 2])
print("肘:", p1, " 手先:", p2)   # 肘 (0,1)、手先 (0.8,1) になるはず

## 到達可能領域

ランダムな関節角で手先位置をプロットすると、半径 $|l_1 - l_2|$ から $l_1 + l_2$ の円環になることを確認します(`kinematics.md` 参照)。

In [ ]:
# 到達可能領域(ワークスペース)を散布図で確認する
rng = np.random.default_rng(0)
qs = rng.uniform([-np.pi, -np.pi], [np.pi, np.pi], size=(3000, 2))
pts = np.array([forward_kinematics(q)[1] for q in qs])
plt.figure(figsize=(5, 5))
plt.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.3)
plt.gca().add_patch(plt.Circle((0, 0), L1 + L2, fill=False, color="r", label="外側の限界 l1+l2"))
plt.gca().add_patch(plt.Circle((0, 0), abs(L1 - L2), fill=False, color="g", label="内側の限界 |l1-l2|"))
plt.axis("equal"); plt.legend(); plt.title("workspace")
plt.show()

## 課題2: 解析的逆運動学

余弦定理から $\cos q_2 = \dfrac{x^2 + y^2 - l_1^2 - l_2^2}{2 l_1 l_2}$。$|\cos q_2| > 1$ なら到達不能です。
$q_2 = \pm \arccos(\cdot)$ の符号が elbow-up / elbow-down の2解に対応します。
$q_1$ は `atan2` を2回使って求めます(単なる `atan` では象限を間違えるので注意)。

In [ ]:
def inverse_kinematics(p, l1=L1, l2=L2, elbow="up"):
    """課題2: 解析的逆運動学。
    手先目標位置 p = [x, y] から [q1, q2] を返す。解が無ければ None"""
    x, y = p
    # ---- ここから課題2 ----
    # (1) 余弦定理で cos(q2) を求める。|cos(q2)| > 1 なら None を返す
    # (2) q2 = ±arccos(...) (elbow が "up" なら負号)
    # (3) q1 = atan2(y, x) - atan2(l2 sin q2, l1 + l2 cos q2)


    # ---- ここまで ----
    return np.array([q1, q2])

# 検算: IKで求めた関節角をFKに入れると元の位置に戻るはず
target = np.array([1.2, 0.6])
for elbow in ["up", "down"]:
    q = inverse_kinematics(target, elbow=elbow)
    _, p2 = forward_kinematics(q)
    print(f"elbow-{elbow}: q = {q}, FK(q) = {p2}, 誤差 = {np.linalg.norm(p2 - target):.2e}")
print("到達不能な点:", inverse_kinematics([3.0, 0.0]))

## 課題3: ヤコビアン

$$J(q) = \begin{pmatrix} -l_1 s_1 - l_2 s_{12} & -l_2 s_{12} \\ l_1 c_1 + l_2 c_{12} & l_2 c_{12} \end{pmatrix}, \qquad \det J = l_1 l_2 \sin q_2$$

実装したら**数値微分と比較**して検算します(REINFORCEの勾配検算と同じ習慣)。$q_2 = 0$(腕が伸び切った姿勢)で行列式が0になる = 特異点であることも数値で確認します。

In [ ]:
def jacobian(q, l1=L1, l2=L2):
    """課題3: 手先速度のヤコビアン (2x2)。dx = J(q) dq"""
    q1, q2 = q
    # ---- ここから課題3 ----
    # kinematics.md の導出を参照。完成したら下の数値微分チェックで検算すること
    J = None
    # ---- ここまで ----
    return J

# 検算: 数値微分と比較する(勾配実装のバグはここで潰す)
q0 = np.array([0.7, -0.4])
J = jacobian(q0)
eps = 1e-6
J_num = np.zeros((2, 2))
for j in range(2):
    dq = np.zeros(2); dq[j] = eps
    J_num[:, j] = (forward_kinematics(q0 + dq)[1] - forward_kinematics(q0 - dq)[1]) / (2 * eps)
print("解析ヤコビアンと数値微分の最大誤差:", np.abs(J - J_num).max())

# 特異点: det J = l1 * l2 * sin(q2)。q2 = 0(伸び切り)で行列式が0になる
for q2 in [0.0, 0.1, np.pi / 2]:
    print(f"q2 = {q2:.2f}: det J = {np.linalg.det(jacobian([0.3, q2])):.4f}")

## 応用: 擬似逆行列による数値IK

$\Delta q = J^+ (p_{\text{target}} - p_{\text{now}})$ を繰り返すと、解析解が使えない場合でもIKが解けます。手先に円を描かせて追従誤差を確認します。

In [ ]:
# 応用: 擬似逆行列による数値IKで手先に円を描かせる
def numerical_ik_step(q, p_target, alpha=1.0):
    """dq = alpha * pinv(J) @ (p_target - p_now) の1ステップ"""
    _, p_now = forward_kinematics(q)
    return q + alpha * np.linalg.pinv(jacobian(q)) @ (p_target - p_now)

ts = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([1.0 + 0.4 * np.cos(ts), 0.4 + 0.4 * np.sin(ts)], axis=1)
q = inverse_kinematics(circle[0])         # 初期姿勢は解析解で
traj, errs = [], []
for p_t in circle:
    for _ in range(5):                     # 各目標点に数回反復で収束させる
        q = numerical_ik_step(q, p_t)
    _, p2 = forward_kinematics(q)
    traj.append(p2); errs.append(np.linalg.norm(p2 - p_t))
traj = np.array(traj)
print(f"追従誤差: max = {max(errs):.2e}")
plt.figure(figsize=(5, 5))
plt.plot(circle[:, 0], circle[:, 1], "r--", label="target")
plt.plot(traj[:, 0], traj[:, 1], "b", label="end effector")
plt.axis("equal"); plt.legend(); plt.title("pseudo-inverse IK tracking")
plt.show()

## 発展課題

1. 特異点近傍(伸び切りに近い目標軌道)で数値IKの関節速度 $\|\Delta q\|$ がどうなるか観察せよ(`kinematics.md` の特異点の節)
2. 到達不能な目標点を与えたとき、擬似逆行列IKはどう振る舞うか
3. リンク長を $l_1 = l_2$ にすると到達可能領域と特異点はどう変わるか

## まとめ

- FK は座標変換の合成、IK は幾何(余弦定理 + atan2)で解ける。IKには複数解・非可解がある
- ヤコビアンは関節速度と手先速度をつなぐ。実装は必ず数値微分で検算する
- 特異点では $\det J = 0$ となり、擬似逆行列IKでも関節速度が大きくなる
- 続きは動力学(`robotics/dynamics.md`)と制御(`robotics/robot-control.md`)へ